<a href="https://colab.research.google.com/github/tnphammy/Microsoft-2B-document-intelligence-agent-harness/blob/main/notebooks/phase-1-extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

## Install libraries


In [1]:
import os
import pathlib
import pandas as pd
!pip install pymupdf
!pip install pymupdf4llm
!pip install python-docx
!pip install 'markitdown[all]'

## Import Data


### Import Files

In [2]:
data_path = '/content/data'

In [3]:
dir_list = os.listdir(data_path)
print("Files and directories in '", data_path, "' :")
# prints all files
print(dir_list)

Files and directories in ' /content/data ' :
['northstar_scholarship_and_financial_aid_guide_2026.pdf', 'northstar_internship_guidelines_2026.docx', 'northstar_student_handbook_and_campus_services_2026.pdf', 'northstar_admissions_and_enrollment_guide_2026.pdf', 'northstar_academic_integrity_policy_2026.pdf', 'northstar_library_services_and_use_rules_2026.pdf', 'northstar_examination_and_grading_policy_2025.pdf', 'northstar_attendance_and_course_participation_policy_2026.docx', 'northstar_laboratory_safety_handbook_2026.docx', 'northstar_examination_and_grading_policy_2026.pdf']


### Sort Files

In [4]:
# Separate list of PDF's and DOCX's
all_pdf = []
all_docx = []

# Sort by PDF and DOCX
for file in dir_list:
  if file.endswith('.pdf'):
    all_pdf.append(file)
  elif file.endswith('.docx'):
    all_docx.append(file)

print(len(all_pdf), " PDFs: ", all_pdf)
print(len(all_docx), " DOCXs: ", all_docx)

7  PDFs:  ['northstar_scholarship_and_financial_aid_guide_2026.pdf', 'northstar_student_handbook_and_campus_services_2026.pdf', 'northstar_admissions_and_enrollment_guide_2026.pdf', 'northstar_academic_integrity_policy_2026.pdf', 'northstar_library_services_and_use_rules_2026.pdf', 'northstar_examination_and_grading_policy_2025.pdf', 'northstar_examination_and_grading_policy_2026.pdf']
3  DOCXs:  ['northstar_internship_guidelines_2026.docx', 'northstar_attendance_and_course_participation_policy_2026.docx', 'northstar_laboratory_safety_handbook_2026.docx']


# Phase 1: Extraction

In [5]:
# Pandas DF to store all extractable information
cols = ["filename", "filepath", "file_type", "md_text", "metadata"]
df = pd.DataFrame(columns=cols)

## PDF's - PyMuPDF



In [6]:
import pymupdf, pymupdf4llm

def extract_pdf_to_entry(pdf:str) -> list:
  """
  Extracts markdown content from pdf file.

  Args:
    pdf: local pdf file

  Returns
    data_entry: a row of information formatted as a list to add to pandas dataframe
  """

  # Get full file path
  file_path = os.path.join(data_path, pdf)

  # Generate markdown
  md_text = pymupdf4llm.to_markdown(file_path)

  # Get metadata
  metadata = pymupdf.open(file_path).metadata

  # Make data entry for this instance in order of: filename, filepath, file_type, md_text, metadata
  data_entry = [pdf, file_path, "pdf", md_text, metadata]

  return data_entry

# Add data_entry as a row in the df
for pdf in all_pdf:
  df.loc[len(df)] = extract_pdf_to_entry(pdf)

display(df.head(7))



=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.


,filename,filepath,file_type,md_text,metadata
0,northstar_scholarship_and_financial_aid_guide_...,/content/data/northstar_scholarship_and_financ...,pdf,**NORTHSTAR UNIVERSITY** | NSU-AID-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
1,northstar_student_handbook_and_campus_services...,/content/data/northstar_student_handbook_and_c...,pdf,**NORTHSTAR UNIVERSITY** | NSU-STUDENT-2026 \...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
2,northstar_admissions_and_enrollment_guide_2026...,/content/data/northstar_admissions_and_enrollm...,pdf,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
3,northstar_academic_integrity_policy_2026.pdf,/content/data/northstar_academic_integrity_pol...,pdf,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
4,northstar_library_services_and_use_rules_2026.pdf,/content/data/northstar_library_services_and_u...,pdf,**NORTHSTAR UNIVERSITY** | NSU-LIB-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
5,northstar_examination_and_grading_policy_2025.pdf,/content/data/northstar_examination_and_gradin...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2025 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
6,northstar_examination_and_grading_policy_2026.pdf,/content/data/northstar_examination_and_gradin...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2026 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."


## DOCX's - python-docx + MarkItDown


In [7]:
from datetime import datetime
from docx import Document
from markitdown import MarkItDown


# Helper function to reformat python datetime objects to our standard format
def format_date(dt):
    if isinstance(dt, datetime):
        # Formats to a PDF-like date string: D:YYYYMMDDHHMMSS
        return f"D:{dt.strftime('%Y%m%d%H%M%S')}"
    return ""

# Iterate through all DOCX's - fill in DataFrame
def extract_docx_to_entry(docx:str) -> list:
  """
  Extracts markdown content from docx file.

  Args:
    docx: local docx file

  Returns
    data_entry: a row of information formatted as a list to add to pandas dataframe
  """
  # 0. Get full filepath
  file_path = os.path.join(data_path, docx)

  # 1. Get markdown text - MarkItDown
  md = MarkItDown()
  result = md.convert(file_path)
  md_text = result.text_content

  # 2.1. Get metadata - python-docx
  doc = Document(file_path) # Pass file_path here, not docx
  props = doc.core_properties

  # 2.2. Reformat metadata
  metadata = {
      "format": "DOCX",  # Explicit file format
      "title": props.title or "",
      "author": props.author or "",
      "subject": props.subject or "",
      "keywords": props.keywords or "",
      "creator": props.last_modified_by or "",  # Fallback to last modifier
      "producer": "python-docx & Microsoft Word",
      "creationDate": format_date(props.created),
      "modDate": format_date(props.modified),
      "trapped": "",
      "encryption": None,  # docx files aren't encrypted natively this way
  }

  # Make data entry for this instance in order of: filename, filepath, file_type, md_text, metadata
  data_entry = [docx, file_path, "docx", md_text, metadata]

  return data_entry

# Add data_entry as a row in the df
for docx in all_docx:
  df.loc[len(df)] = extract_docx_to_entry(docx)


display(df.tail(3))

,filename,filepath,file_type,md_text,metadata
7,northstar_internship_guidelines_2026.docx,/content/data/northstar_internship_guidelines_...,docx,**Undergraduate Internship Guidelines**\n\n**A...,"{'format': 'DOCX', 'title': 'Undergraduate Int..."
8,northstar_attendance_and_course_participation_...,/content/data/northstar_attendance_and_course_...,docx,**Attendance and Course Participation Policy**...,"{'format': 'DOCX', 'title': 'Attendance and Co..."
9,northstar_laboratory_safety_handbook_2026.docx,/content/data/northstar_laboratory_safety_hand...,docx,**Undergraduate Laboratory Safety Handbook**\n...,"{'format': 'DOCX', 'title': 'Undergraduate Lab..."


In [8]:
# Examine current Dataframe
display(df.head(10))

,filename,filepath,file_type,md_text,metadata
0,northstar_scholarship_and_financial_aid_guide_...,/content/data/northstar_scholarship_and_financ...,pdf,**NORTHSTAR UNIVERSITY** | NSU-AID-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
1,northstar_student_handbook_and_campus_services...,/content/data/northstar_student_handbook_and_c...,pdf,**NORTHSTAR UNIVERSITY** | NSU-STUDENT-2026 \...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
2,northstar_admissions_and_enrollment_guide_2026...,/content/data/northstar_admissions_and_enrollm...,pdf,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
3,northstar_academic_integrity_policy_2026.pdf,/content/data/northstar_academic_integrity_pol...,pdf,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
4,northstar_library_services_and_use_rules_2026.pdf,/content/data/northstar_library_services_and_u...,pdf,**NORTHSTAR UNIVERSITY** | NSU-LIB-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
5,northstar_examination_and_grading_policy_2025.pdf,/content/data/northstar_examination_and_gradin...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2025 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
6,northstar_examination_and_grading_policy_2026.pdf,/content/data/northstar_examination_and_gradin...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2026 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
7,northstar_internship_guidelines_2026.docx,/content/data/northstar_internship_guidelines_...,docx,**Undergraduate Internship Guidelines**\n\n**A...,"{'format': 'DOCX', 'title': 'Undergraduate Int..."
8,northstar_attendance_and_course_participation_...,/content/data/northstar_attendance_and_course_...,docx,**Attendance and Course Participation Policy**...,"{'format': 'DOCX', 'title': 'Attendance and Co..."
9,northstar_laboratory_safety_handbook_2026.docx,/content/data/northstar_laboratory_safety_hand...,docx,**Undergraduate Laboratory Safety Handbook**\n...,"{'format': 'DOCX', 'title': 'Undergraduate Lab..."


# Phase 2: Chunk Text

# Phase 3: Embeddings

# Phase 4: Depluplication & Verification